# Laboratório — Derivadas locais e grafo computacional

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/04-deep-learning/m5-redes-neurais-do-zero/notebooks/08-derivadas-locais-grafo-computacional-laboratorio.ipynb)

**Modo:** tutorial reproduzível · **implementação:** NumPy puro · **seed:** `20260909`

Este laboratório implementa VJPs e uma fita reversa mínima para operações elemento a elemento. Não usa PyTorch, TensorFlow, JAX nem autograd. A camada afim fica para a Aula 09.

## Goal

Ao final, você terá uma evidência executável de que:

1. o gradiente downstream combina derivada local e upstream;
2. ramificações somam contribuições;
3. um VJP evita construir Jacobianas desnecessárias;
4. broadcasting precisa ser desfeito no backward;
5. uma passagem reversa mínima obedece à ordem topológica;
6. diferenças centrais e o teste de adjunção detectam erros.

## Setup

### Ambiente e contrato numérico

- Python `>=3.11`
- NumPy `>=1.26`
- Matplotlib `>=3.8`
- nbformat `>=5.9` apenas para validar o arquivo
- `float64` para tornar o gradient checking mais sensível

Os dados são pequenos arrays sintéticos, declarados no próprio notebook. Não há download, credencial nem estado externo.

In [ ]:
import platform
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260909
DTYPE = np.float64
rng = np.random.default_rng(SEED)

assert tuple(map(int, np.__version__.split(".")[:2])) >= (1, 26)
assert tuple(map(int, matplotlib.__version__.split(".")[:2])) >= (3, 8)

print({
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "Matplotlib": matplotlib.__version__,
    "seed": SEED,
    "dtype": str(np.dtype(DTYPE)),
})

## Steps

### 1. Forward de um grafo ramificado

Usaremos

$$
L=\sum_i c_i^2,\qquad
\mathbf c=\mathbf x\odot\mathbf y+\sin\mathbf x.
$$

O vetor $\mathbf x$ participa do produto e do seno. Portanto, seu gradiente deverá somar duas rotas.

In [ ]:
x = np.array([0.5, -1.0, 2.0], dtype=DTYPE)
y = np.array([3.0, -2.0, 0.25], dtype=DTYPE)

a = x * y
b = np.sin(x)
c = a + b
d = c**2
loss = float(np.sum(d))

print({
    "a=x*y": np.round(a, 9),
    "b=sin(x)": np.round(b, 9),
    "c=a+b": np.round(c, 9),
    "L=sum(c²)": loss,
})

### 2. Backward manual: local × upstream

Começamos com $\bar L=1$. A redução por soma replica esse upstream, o quadrado multiplica por $2\mathbf c$, e a soma distribui a mensagem. No nó $\mathbf x$, as rotas do produto e do seno precisam ser acumuladas.

In [ ]:
grad_loss = 1.0
grad_d = np.ones_like(d) * grad_loss
grad_c = grad_d * (2.0 * c)
grad_a = grad_c
grad_b = grad_c

grad_x_via_product = grad_a * y
grad_x_via_sine = grad_b * np.cos(x)
grad_x_manual = grad_x_via_product + grad_x_via_sine
grad_y_manual = grad_a * x

print("grad_x via produto:", np.round(grad_x_via_product, 9))
print("grad_x via seno:   ", np.round(grad_x_via_sine, 9))
print("grad_x acumulado:  ", np.round(grad_x_manual, 9))
print("grad_y:            ", np.round(grad_y_manual, 9))

### 3. Visualizar as contribuições

O gráfico não representa importância de feature. Ele apenas mostra, para este ponto, quanto cada rota do grafo contribui para $\partial L/\partial x_i$.

In [ ]:
indices = np.arange(x.size)
width = 0.36
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(indices - width / 2, grad_x_via_product, width, label="via produto")
ax.bar(indices + width / 2, grad_x_via_sine, width, label="via seno")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set(
    title="Contribuições acumuladas no gradiente de x",
    xlabel="coordenada i",
    ylabel="contribuição para ∂L/∂xᵢ",
    xticks=indices,
)
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

print("Texto alternativo: barras agrupadas mostram as contribuições das rotas produto e seno; "
      "a soma vertical por coordenada forma o gradiente total de x.")

### 4. Diferenças centrais como oráculo

Para uma função escalar $L(\mathbf x)$, perturbamos uma coordenada por vez:

$$
\frac{\partial L}{\partial x_j}\approx
\frac{L(\mathbf x+\varepsilon\mathbf e_j)-L(\mathbf x-\varepsilon\mathbf e_j)}{2\varepsilon}.
$$

Isso é uma validação aproximada, não o algoritmo de treino.

In [ ]:
def scalar_loss(x_value, y_value):
    c_value = x_value * y_value + np.sin(x_value)
    return float(np.sum(c_value**2))


def central_gradient(function, value, epsilon=1e-6):
    value = np.asarray(value, dtype=DTYPE)
    gradient = np.zeros_like(value)
    for index in np.ndindex(value.shape):
        plus = value.copy()
        minus = value.copy()
        plus[index] += epsilon
        minus[index] -= epsilon
        gradient[index] = (function(plus) - function(minus)) / (2.0 * epsilon)
    return gradient


grad_x_numeric = central_gradient(lambda candidate: scalar_loss(candidate, y), x)
grad_y_numeric = central_gradient(lambda candidate: scalar_loss(x, candidate), y)
error_x = float(np.max(np.abs(grad_x_manual - grad_x_numeric)))
error_y = float(np.max(np.abs(grad_y_manual - grad_y_numeric)))

print(f"erro máximo em x={error_x:.3e}")
print(f"erro máximo em y={error_y:.3e}")

### 5. VJP com Jacobiana visível apenas para estudo

Defina

$$
f(\mathbf z)=
\begin{bmatrix}
z_0z_1\
\sin z_0+z_1^2
\end{bmatrix}.
$$

Para um upstream $\mathbf v$, queremos $J_f(\mathbf z)^\top\mathbf v$. Construiremos a Jacobiana apenas neste exemplo $2\times2$ para conferir o VJP direto.

In [ ]:
def vector_function(z):
    z = np.asarray(z, dtype=DTYPE)
    if z.shape != (2,):
        raise ValueError("z deve ter shape (2,)")
    return np.array([z[0] * z[1], np.sin(z[0]) + z[1]**2], dtype=DTYPE)


def vector_vjp(z, upstream):
    z = np.asarray(z, dtype=DTYPE)
    upstream = np.asarray(upstream, dtype=DTYPE)
    if z.shape != (2,) or upstream.shape != (2,):
        raise ValueError("z e upstream devem ter shape (2,)")
    return np.array([
        upstream[0] * z[1] + upstream[1] * np.cos(z[0]),
        upstream[0] * z[0] + upstream[1] * 2.0 * z[1],
    ])


z = np.array([0.7, -1.2], dtype=DTYPE)
v = np.array([1.5, -0.4], dtype=DTYPE)
jacobian = np.array([
    [z[1], z[0]],
    [np.cos(z[0]), 2.0 * z[1]],
])
vjp_explicit = jacobian.T @ v
vjp_direct = vector_vjp(z, v)
vjp_error = float(np.max(np.abs(vjp_explicit - vjp_direct)))

print("J explícita:\n", np.round(jacobian, 9))
print("Jᵀv explícito:", np.round(vjp_explicit, 9))
print("VJP direto:   ", np.round(vjp_direct, 9))
print(f"erro={vjp_error:.3e}")

### 6. Teste de adjunção sem materializar $J$

Para uma direção $\mathbf u$:

$$
\mathbf v^\top(J\mathbf u)=(J^\top\mathbf v)^\top\mathbf u.
$$

Aproximamos $J\mathbf u$ por uma diferença central direcional.

In [ ]:
u = rng.normal(size=2).astype(DTYPE)
epsilon = 1e-6
jvp_numeric = (vector_function(z + epsilon * u) - vector_function(z - epsilon * u)) / (2.0 * epsilon)
left_adjoint = float(v @ jvp_numeric)
right_adjoint = float(vjp_direct @ u)
adjoint_error = abs(left_adjoint - right_adjoint)

print({
    "u": np.round(u, 9),
    "vᵀ(Ju)": left_adjoint,
    "(Jᵀv)ᵀu": right_adjoint,
    "erro": adjoint_error,
})

### 7. Desfazer broadcasting

No forward, `X + b` replica conceitualmente `b` por todas as linhas. No backward, as contribuições destinadas a `b` precisam ser somadas até recuperar seu shape original.

In [ ]:
def unbroadcast(gradient, original_shape):
    gradient = np.asarray(gradient, dtype=DTYPE)
    original_shape = tuple(original_shape)
    if gradient.ndim < len(original_shape):
        raise ValueError("gradiente não pode ter menos eixos que a entrada original")

    while gradient.ndim > len(original_shape):
        gradient = gradient.sum(axis=0)

    for axis, size in enumerate(original_shape):
        if size == 1 and gradient.shape[axis] != 1:
            gradient = gradient.sum(axis=axis, keepdims=True)
        elif size != gradient.shape[axis]:
            raise ValueError("shape incompatível com o broadcasting registrado")

    result = gradient.reshape(original_shape)
    if result.shape != original_shape:
        raise AssertionError("unbroadcast não recuperou o shape original")
    return result


batch_gradient = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)
grad_bias = unbroadcast(batch_gradient, (3,))
grad_row_bias = unbroadcast(batch_gradient, (1, 3))
grad_scalar = unbroadcast(batch_gradient, ())

print({
    "grad_bias_(3,)": grad_bias,
    "grad_bias_(1,3)": grad_row_bias,
    "grad_scalar": grad_scalar,
})

### 8. Contraprova: copiar não substitui reduzir

Se o upstream de `Y = X + b` é a matriz acima, usar apenas a primeira linha para `grad_b` descarta a contribuição do segundo exemplo.

In [ ]:
wrong_grad_bias = batch_gradient[0].copy()
correct_grad_bias = batch_gradient.sum(axis=0)
discarded_contribution = correct_grad_bias - wrong_grad_bias

print("errado, primeira linha:", wrong_grad_bias)
print("correto, soma do lote:", correct_grad_bias)
print("contribuição descartada:", discarded_contribution)

### 9. Uma fita reversa mínima

O objeto abaixo registra dados, pais e uma função VJP. As operações aceitas são soma, produto elemento a elemento, seno, quadrado e soma total. Não há produto matricial: o backward da camada afim pertence à próxima aula.

In [ ]:
class Node:
    def __init__(self, data, parents=(), vjp=None, name=""):
        self.data = np.array(data, dtype=DTYPE, copy=True)
        self.parents = tuple(parents)
        self.vjp = vjp
        self.name = name or "node"
        self.grad = np.zeros_like(self.data)


def add(left, right, name="add"):
    output = left.data + right.data
    def vjp(upstream):
        return (
            unbroadcast(upstream, left.data.shape),
            unbroadcast(upstream, right.data.shape),
        )
    return Node(output, (left, right), vjp, name)


def multiply(left, right, name="multiply"):
    output = left.data * right.data
    left_cache = left.data.copy()
    right_cache = right.data.copy()
    def vjp(upstream):
        return (
            unbroadcast(upstream * right_cache, left.data.shape),
            unbroadcast(upstream * left_cache, right.data.shape),
        )
    return Node(output, (left, right), vjp, name)


def sine(value, name="sine"):
    input_cache = value.data.copy()
    return Node(np.sin(input_cache), (value,),
                lambda upstream: (upstream * np.cos(input_cache),), name)


def square(value, name="square"):
    input_cache = value.data.copy()
    return Node(input_cache**2, (value,),
                lambda upstream: (upstream * 2.0 * input_cache,), name)


def sum_all(value, name="sum"):
    input_shape = value.data.shape
    return Node(np.sum(value.data), (value,),
                lambda upstream: (np.ones(input_shape, dtype=DTYPE) * upstream,), name)


def backward(output, upstream=None):
    topological = []
    visited = set()

    def visit(node):
        if id(node) in visited:
            return
        visited.add(id(node))
        for parent in node.parents:
            visit(parent)
        topological.append(node)

    visit(output)
    if upstream is None:
        if output.data.shape != ():
            raise ValueError("saída vetorial exige upstream explícito")
        seed = np.array(1.0, dtype=DTYPE)
    else:
        upstream = np.asarray(upstream, dtype=DTYPE)
        if upstream.shape != output.data.shape:
            raise ValueError("upstream deve ter o shape da saída")
        seed = upstream.copy()

    for node in topological:
        node.grad = np.zeros_like(node.data)
    output.grad = seed

    reverse_order = []
    for node in reversed(topological):
        reverse_order.append(node.name)
        if node.vjp is None:
            continue
        contributions = node.vjp(node.grad)
        if len(contributions) != len(node.parents):
            raise AssertionError("VJP deve devolver uma contribuição por pai")
        for parent, contribution in zip(node.parents, contributions):
            contribution = np.asarray(contribution, dtype=DTYPE)
            if contribution.shape != parent.data.shape:
                raise AssertionError("gradiente não recuperou o shape do pai")
            parent.grad += contribution
    return reverse_order

### 10. Reproduzir o grafo manual

A fita deve retornar os mesmos gradientes derivados anteriormente e percorrer a loss antes de suas dependências.

In [ ]:
x_node = Node(x, name="x")
y_node = Node(y, name="y")
a_node = multiply(x_node, y_node, name="a=x*y")
b_node = sine(x_node, name="b=sin(x)")
c_node = add(a_node, b_node, name="c=a+b")
d_node = square(c_node, name="d=c²")
loss_node = sum_all(d_node, name="L=sum(d)")

reverse_order = backward(loss_node)
engine_error_x = float(np.max(np.abs(x_node.grad - grad_x_manual)))
engine_error_y = float(np.max(np.abs(y_node.grad - grad_y_manual)))

print("ordem reversa:", " -> ".join(reverse_order))
print("grad x da fita:", np.round(x_node.grad, 9))
print("grad y da fita:", np.round(y_node.grad, 9))
print({"erro_x": engine_error_x, "erro_y": engine_error_y})

### 11. Ramificação: acumular, não sobrescrever

Para $L=x^2+x$ em $x=3$, os caminhos contribuem com 6 e 1. O gradiente correto é 7.

In [ ]:
branch_x = Node(3.0, name="x")
branch_square = square(branch_x, name="x²")
branch_loss = add(branch_square, branch_x, name="L=x²+x")
branch_order = backward(branch_loss, upstream=np.array(1.0))

wrong_overwrite_candidates = np.array([2.0 * 3.0, 1.0])
print({
    "contribuições": wrong_overwrite_candidates,
    "gradiente_acumulado": float(branch_x.grad),
    "possíveis_sobrescritas_erradas": wrong_overwrite_candidates,
    "ordem": branch_order,
})

### 12. Saída vetorial exige upstream explícito

Para $\mathbf q=\mathbf r^2+\sin\mathbf r$, o vetor upstream escolhe a projeção escalar cuja derivada será calculada.

In [ ]:
vector_input = Node(np.array([0.2, -0.7, 1.1]), name="r")
vector_output = add(square(vector_input, name="r²"),
                    sine(vector_input, name="sin(r)"), name="q")
vector_upstream = np.array([1.0, -0.5, 2.0], dtype=DTYPE)
backward(vector_output, upstream=vector_upstream)
vector_expected = vector_upstream * (2.0 * vector_input.data + np.cos(vector_input.data))
vector_seed_error = float(np.max(np.abs(vector_input.grad - vector_expected)))

missing_upstream_rejected = False
try:
    backward(vector_output)
except ValueError:
    missing_upstream_rejected = True

print("VJP da saída vetorial:", np.round(vector_input.grad, 9))
print(f"erro={vector_seed_error:.3e}; upstream ausente rejeitado={missing_upstream_rejected}")

### 13. Broadcasting dentro da fita

Agora verificamos que a soma de uma matriz com um vetor devolve gradientes com os shapes originais.

In [ ]:
matrix_node = Node(np.arange(6, dtype=DTYPE).reshape(2, 3), name="X")
bias_node = Node(np.array([0.1, -0.2, 0.3]), name="b")
broadcast_output = add(matrix_node, bias_node, name="Y=X+b")
broadcast_loss = sum_all(broadcast_output, name="sum(Y)")
backward(broadcast_loss)

print({
    "shape_grad_X": matrix_node.grad.shape,
    "shape_grad_b": bias_node.grad.shape,
    "grad_X": matrix_node.grad,
    "grad_b": bias_node.grad,
})

## Checks

Os testes abaixo cobrem valores manuais, gradient checking, VJP, adjunção, broadcasting, topologia, acumulação, seed vetorial e falhas intencionais.

In [ ]:
# Forward e backward manual
assert np.isclose(loss, 7.246434179178013, atol=1e-12)
assert a.shape == b.shape == c.shape == d.shape == (3,)
assert grad_x_manual.shape == x.shape
assert grad_y_manual.shape == y.shape
assert np.allclose(grad_x_manual, 2.0 * c * (y + np.cos(x)))
assert np.allclose(grad_y_manual, 2.0 * c * x)
assert error_x < 2e-8
assert error_y < 2e-8

# VJP e adjunção
assert jacobian.shape == (2, 2)
assert vjp_direct.shape == z.shape
assert vjp_error < 1e-14
assert adjoint_error < 2e-9
assert np.isfinite(jvp_numeric).all()

# Unbroadcast
assert grad_bias.shape == (3,)
assert grad_row_bias.shape == (1, 3)
assert grad_scalar.shape == ()
assert np.array_equal(grad_bias, np.array([5.0, 7.0, 9.0]))
assert np.array_equal(grad_row_bias, np.array([[5.0, 7.0, 9.0]]))
assert float(grad_scalar) == 21.0
assert np.array_equal(discarded_contribution, batch_gradient[1])

# Fita reversa e acumulação
assert reverse_order[0] == "L=sum(d)"
assert reverse_order[-1] == "x"
assert engine_error_x == 0.0
assert engine_error_y == 0.0
assert np.isclose(float(branch_x.grad), 7.0)
assert len(set(branch_order)) == len(branch_order)
assert vector_seed_error == 0.0
assert missing_upstream_rejected
assert matrix_node.grad.shape == (2, 3)
assert bias_node.grad.shape == (3,)
assert np.array_equal(matrix_node.grad, np.ones((2, 3)))
assert np.array_equal(bias_node.grad, np.array([2.0, 2.0, 2.0]))

# Falhas intencionais de contrato
try:
    unbroadcast(np.ones((2, 4)), (3,))
except ValueError:
    pass
else:
    raise AssertionError("shape incompatível deveria ser rejeitado")

try:
    backward(vector_output, upstream=np.ones(2))
except ValueError:
    pass
else:
    raise AssertionError("upstream com shape incorreto deveria ser rejeitado")

print("35 contratos verificados com sucesso.")

## Resultados confirmados

Após executar todas as células:

- a fita e a derivação manual coincidem exatamente para os dois vetores de entrada;
- diferenças centrais concordam com os gradientes analíticos em erro inferior a `2e-8`;
- o VJP direto coincide com $J^\top\mathbf v$ explícito até precisão de máquina;
- o teste de adjunção concorda em erro inferior a `2e-9`;
- a ramificação $x^2+x$ acumula `7.0` em $x=3$;
- `unbroadcast` reduz uma matriz `(2, 3)` para gradientes `(3,)`, `(1, 3)` e escalar;
- 35 contratos são verificados sem frameworks de diferenciação automática.

## Next Steps

Na Aula 09, aplicaremos esta interface VJP à camada afim $Z=XW+b$. Derivaremos os três gradientes, rastrearemos os shapes e confirmaremos os produtos matriciais com diferenças centrais, ainda em NumPy puro.